# 01 — Exploratory Data Analysis

Diabetes Risk Prediction — CDC Diabetes Health Indicators (BRFSS 2015)

This notebook loads the raw data through `src.data_loader`, validates it, and explores class balance, distributions, and correlations. Reusable logic lives in `src/`; this notebook is the narrative layer.

In [ ]:
import sys
from pathlib import Path

# Allow imports from src/ when running notebooks from the notebooks/ folder
sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.data_loader import load_raw_data
from src.utils import load_config, set_seed

config = load_config()
set_seed(config["random_seed"])
sns.set_theme(style="whitegrid")

In [ ]:
df = load_raw_data()
print(df.shape)
df.head()

## Class balance

In [ ]:
target = config["data"]["target_column"]
counts = df[target].value_counts(normalize=True) * 100
print(counts)

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x=df[target], ax=ax)
ax.set_title("Class Balance — Diabetes_binary")
ax.set_xticklabels(["No Diabetes", "Prediabetes/Diabetes"])
plt.show()

## Missing values and basic quality checks

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe()

## Key distributions by diabetes status

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.boxplot(x=target, y="BMI", data=df, ax=axes[0])
axes[0].set_title("BMI by Diabetes Status")

sns.countplot(x="GenHlth", hue=target, data=df, ax=axes[1])
axes[1].set_title("General Health by Diabetes Status")

sns.countplot(x="Age", hue=target, data=df, ax=axes[2])
axes[2].set_title("Age Bucket by Diabetes Status")
axes[2].tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.savefig("../reports/figures/eda_key_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## Correlation heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, ax=ax)
ax.set_title("Feature Correlation Heatmap")
plt.savefig("../reports/figures/eda_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top correlations with target:")
print(corr[target].sort_values(ascending=False))

## Notes / next steps

- Record 2-3 headline observations here after running this notebook (e.g. strongest correlated features, any surprising distribution).
- Proceed to `02_statistical_analysis.ipynb` to formally test which features are significantly associated with diabetes status.